In [2]:
# IMPORTANT: set these BEFORE importing NumPy, SciPy or Gensim
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def english_tokens(text):
    text = text.lower()
    return re.findall(r"[a-z]+", text)

def hindi_tokens(text):
    text = normalize_unicode(text)
    return re.findall(r"[\u0900-\u097F]+", text)

def average_vectors(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

print("Core libraries loaded.")
print("BLAS threads limited to 1.")


Core libraries loaded.
BLAS threads limited to 1.


In [3]:
# Re-create the Q1 CBOW model so this notebook can run independently.
sentences = ['machine learning models learn patterns from data', 'machine learning algorithms improve prediction accuracy', 'deep learning models use neural networks', 'neural networks learn complex patterns from data', 'artificial intelligence systems solve complex problems', 'artificial intelligence uses machine learning methods', 'data science combines statistics and programming', 'data analysis helps researchers find useful patterns', 'data scientists build models from large datasets', 'researchers analyze data to test new ideas', 'computer vision systems analyze images', 'computer vision models detect objects in images', 'image classification assigns labels to images', 'deep learning improves image classification performance', 'natural language processing analyzes human language', 'language models learn patterns from text', 'natural language systems process words and sentences', 'text classification assigns categories to documents', 'students learn programming in computer science courses', 'students practice algorithms and data structures', 'programming skills help students solve problems', 'computer science students study machine learning', 'online courses provide lectures and programming exercises', 'online education helps students learn new skills', 'education platforms use data to track student progress', 'teachers use data analysis to understand learning outcomes', 'algorithms process data to produce useful results', 'algorithms can classify images and text', 'models learn from training data', 'training data improves machine learning models', 'large datasets help neural networks learn', 'researchers train models on large datasets', 'prediction models estimate future outcomes', 'machine learning supports prediction in many applications', 'artificial intelligence research develops new models', 'researchers develop algorithms for language processing', 'language models represent words using their contexts', 'words with similar contexts can have related meanings', 'context helps a model represent the meaning of words', 'distributional representations describe words through context', 'data models support intelligent applications', 'machine learning supports computer vision', 'neural networks support language processing', 'researchers compare models using data', 'students learn artificial intelligence and machine learning', 'software engineers build data driven applications', 'modern systems use algorithms to process information', 'prediction models help organizations make decisions', 'technology companies develop intelligent software', 'computer science research studies learning algorithms']
tokenized_corpus = [english_tokens(s) for s in sentences]

cbow_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=0,
    epochs=100,
    seed=42
)

print("CBOW vocabulary size:", len(cbow_model.wv))


CBOW vocabulary size: 133


In [4]:
# Q2(a): Load pre-trained GloVe vectors.
# glove-wiki-gigaword-50 provides 50-dimensional English GloVe vectors.
import gensim.downloader as api

print("Loading GloVe. The first download may take some time...")
glove = api.load("glove-wiki-gigaword-50")

print("GloVe loaded.")
print("GloVe vector dimensionality:", glove.vector_size)


Loading GloVe. The first download may take some time...
[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe loaded.
GloVe vector dimensionality: 50


In [5]:
# Alternative local loading method if a GloVe .txt/.vec file is already available.
# Uncomment and set the correct path if needed.
#
# from gensim.models import KeyedVectors
# glove = KeyedVectors.load_word2vec_format(
#     r"C:\path\to\glove.6B.50d.w2v.txt",
#     binary=False
# )


In [6]:
# Q2(b): Identify at least 10 words common to CBOW and GloVe.
candidate_words = [
    "machine", "learning", "data", "model", "models", "neural",
    "network", "networks", "language", "computer", "science",
    "students", "algorithms", "text", "researchers"
]

common_words = [w for w in candidate_words if w in cbow_model.wv and w in glove]
print("Common words:", common_words)
print("Number of common words:", len(common_words))


Common words: ['machine', 'learning', 'data', 'model', 'models', 'neural', 'networks', 'language', 'computer', 'science', 'students', 'algorithms', 'text', 'researchers']
Number of common words: 14


In [7]:
# Q2(c): Display GloVe vector dimensionality and sample vectors.
glove_sample_words = common_words[:10]
glove_vectors = pd.DataFrame(
    [glove[w] for w in glove_sample_words],
    index=glove_sample_words
)

print("GloVe dimensions:", glove.vector_size)
display(glove_vectors.round(4))


GloVe dimensions: 50


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
machine,-0.3416,-0.8127,1.4513,0.0591,-0.0808,0.3957,0.1006,-0.5468,-0.1889,0.1136,...,1.0191,0.0159,-0.7532,0.5899,0.4546,-0.1968,0.4214,-0.4317,0.1198,0.1409
learning,0.2046,0.4866,-0.5531,-0.2702,0.2634,0.1575,-0.2899,-0.5182,0.0518,0.3622,...,0.0883,-0.9178,-0.3667,0.9926,0.2185,-0.3160,1.2030,0.2699,-0.1409,0.7078
data,0.5310,-0.5587,1.7674,0.4482,0.2234,-0.3456,-0.7768,-0.9612,1.1669,0.0743,...,0.7432,-0.4503,0.1941,0.4861,-0.4308,-0.2521,1.2774,1.5815,0.6584,-0.2098
model,0.3509,0.6708,0.3862,-0.1228,-0.0111,0.9600,-0.4323,-1.9641,-0.1668,-0.0270,...,0.2975,-0.3884,0.3121,-0.3408,-0.3548,-0.2820,0.7158,-0.2907,0.2291,0.6269
models,0.7987,0.3157,0.7207,-0.1929,-0.4816,0.7944,-0.5073,-2.2100,-0.4856,0.1890,...,0.1464,-0.3693,0.5650,-0.0982,0.0496,-0.2249,0.9960,-0.0365,-0.1041,0.0813
neural,0.9280,0.2910,0.6784,1.0444,-0.7255,2.1995,0.8877,-0.9478,0.6743,0.2491,...,-0.5255,-0.9314,-1.0553,1.4401,0.3081,-0.8487,1.9986,0.1079,-0.2363,-0.1798
networks,1.1983,-0.3076,1.5738,1.5511,-0.8385,0.3251,-0.5706,-0.8669,0.4037,0.4004,...,0.6600,-0.0880,-0.3522,0.9233,-0.4062,-0.1574,-0.2378,0.2078,-0.3240,0.6126
language,-0.5799,-0.1101,-1.1557,-0.0030,-0.2061,0.4529,-0.1667,-1.0382,-0.9924,0.3988,...,0.8263,0.5709,0.2120,0.4686,-0.6003,0.2992,0.6794,1.4238,-0.0322,-0.1260
computer,0.0791,-0.8150,1.7901,0.9165,0.1080,-0.5563,-0.8443,-1.4951,0.1342,0.6363,...,0.6644,0.0551,0.0379,1.3275,0.3099,0.5070,1.2357,0.1274,-0.1143,0.2071
science,-0.6033,1.0577,-0.3045,0.0567,0.0006,-0.1587,-0.3934,-1.6816,0.2575,0.0965,...,-0.0250,-0.4686,0.2562,0.3751,-0.1243,0.7288,0.9060,0.8355,0.0002,0.4476


In [8]:
# Q2(d): GloVe cosine similarity for at least five word pairs.
pairs = [
    ("machine", "learning"),
    ("neural", "network"),
    ("computer", "science"),
    ("language", "text"),
    ("data", "model")
]

results = []
for a, b in pairs:
    if a in glove and b in glove:
        results.append([a, b, float(glove.similarity(a, b))])

display(pd.DataFrame(
    results,
    columns=["Word 1", "Word 2", "GloVe Cosine Similarity"]
).round(4))


,Word 1,Word 2,GloVe Cosine Similarity
0,machine,learning,0.4803
1,neural,network,0.4214
2,computer,science,0.6250
3,language,text,0.7400
4,data,model,0.5245


In [9]:
# Q2(e): Compare nearest neighbours from CBOW and GloVe.
comparison_words = ["machine", "learning", "data", "language", "computer"]

for word in comparison_words:
    if word in cbow_model.wv and word in glove:
        print(f"\\nWORD: {word}")
        print("CBOW :", [x[0] for x in cbow_model.wv.most_similar(word, topn=5)])
        print("GloVe:", [x[0] for x in glove.most_similar(word, topn=5)])


\nWORD: machine
CBOW : ['the', 'words', 'to', 'prediction', 'science']
GloVe: ['machines', 'device', 'using', 'gun', 'used']
\nWORD: learning
CBOW : ['model', 'helps', 'models', 'analysis', 'algorithms']
GloVe: ['teaching', 'skills', 'experience', 'practical', 'knowledge']
\nWORD: data
CBOW : ['can', 'track', 'to', 'use', 'helps']
GloVe: ['information', 'tracking', 'database', 'analysis', 'applications']
\nWORD: language
CBOW : ['to', 'models', 'contexts', 'the', 'represent']
GloVe: ['languages', 'word', 'spoken', 'vocabulary', 'translation']
\nWORD: computer
CBOW : ['researchers', 'models', 'images', 'algorithms', 'words']
GloVe: ['computers', 'software', 'technology', 'electronic', 'internet']


### Q2(f): Word2Vec vs GloVe

**Word2Vec CBOW**
- Learns embeddings by predicting a target word from its local context.
- Training is based on local context windows.
- The resulting vectors capture useful semantic and syntactic relationships.

**GloVe**
- Uses global word-word co-occurrence statistics from a corpus.
- The objective is based on relationships in the global co-occurrence matrix.
- The pre-trained GloVe vectors used here were learned from a much larger corpus than the small CBOW corpus.

Therefore, the two models can produce different nearest neighbours even for the same word.


### Q2(g): Factors affecting embeddings

- **Corpus size:** larger and more diverse corpora provide more contextual evidence.
- **Vocabulary:** determines which words can receive vectors.
- **Vector dimension:** larger dimensions can encode more information but increase computation and may require more data.
- **Training method:** local predictive training and global co-occurrence training capture different statistical signals.
- **Hyperparameters:** context window, epochs, minimum count and optimization choices affect the learned space.


In [10]:
# Q2(h) Optional extension: simple analogy experiment with GloVe.
# This is attempted only when all analogy words exist.
analogy = ["king", "man", "woman"]
if all(w in glove for w in analogy):
    print(glove.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5
    ))
else:
    print("Analogy words are not all available in the loaded model.")


[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172), ('daughter', 0.7473883628845215), ('elizabeth', 0.7460219860076904)]
